## Task 2 :  Sentiment Analysis Model

### Name : Prem Vikas Palkar
### Intern ID : IN226105802

In [1]:
%pip install nltk

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [43]:
# Import all require libraris
import re
import nltk
import pandas as pd

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize     # For tokenization (better than text.split())
from nltk.stem import WordNetLemmatizer     # For Lemmatization, to convert word into its root form

from sklearn.feature_extraction.text import CountVectorizer     # For BAG of words
from sklearn.feature_extraction.text import TfidfVectorizer     # For TF-IDF

# For model building
from sklearn.model_selection import train_test_split 
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier

# For Model Evalution
from sklearn.metrics import accuracy_score, classification_report

In [8]:
# Download stopwords
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt_tab')      # Tokenizer uses punlt model for tokenization so we need to download it seperately

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\premv\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\premv\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\premv\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.


True

In [4]:
# Get stopwords
stop_words = set(stopwords.words('english'))    # We use set for stopwords for faster lookups O(1), as compared to list we need to look up one by one O(n), but not for set.
stop_words.remove("no")
stop_words.remove("not")

# Create Lemmatizer object
lemmatizer = WordNetLemmatizer()

In [5]:
# Read the IMDb csv files and check some basic info
df =  pd.read_csv("IMDB Dataset.csv")

print("Data : \n",df.head())
print("\nShape : ",df.shape)
print("\nColumns: ", df.columns)
print("\nClass Distribution : \n", df['sentiment'].value_counts())

Data : 
                                               review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive

Shape :  (50000, 2)

Columns:  Index(['review', 'sentiment'], dtype='str')

Class Distribution : 
 sentiment
positive    25000
negative    25000
Name: count, dtype: int64


#### Preprocessing function 

In [10]:
def preprocess_text(text):
    
    # lowercasing
    text = text.lower()
    
    # Removing URL's
    text = re.sub(r'http\S+|www\S+', '', text)
    
    # Remove Punctuation
    text = text = re.sub(r'[^\w\s]', '', text)
    
    # Tokenize
    tokens = word_tokenize(text)
    
    # Remove stopwords
    tokens = [word for word in tokens if word not in stop_words]
    
    # Lemmatize
    tokens = [lemmatizer.lemmatize(word) for word in tokens]
    
    # Join back sentences
    text = " ".join(tokens)
        
    return text

In [11]:
text = "I am not happy with this movie!!!"
print(preprocess_text(text))

not happy movie


#### New Column for Processed Text

In [ ]:
# Using pandas .apply() to apply function to every row as the data is huge, loops must not be used
df['cleaned_review'] = df['review'].apply(preprocess_text)

In [17]:
df[['review','cleaned_review']].head()

,review,cleaned_review
0,One of the other reviewers has mentioned that ...,one reviewer mentioned watching 1 oz episode y...
1,A wonderful little production. <br /><br />The...,wonderful little production br br filming tech...
2,I thought this was a wonderful way to spend ti...,thought wonderful way spend time hot summer we...
3,Basically there's a family where a little boy ...,basically there family little boy jake think t...
4,"Petter Mattei's ""Love in the Time of Money"" is...",petter matteis love time money visually stunni...


#### Feature Engineering : Text --> Numbers

#### 1. Bag Of Words (BOW)

In [ ]:
# Bag Of Words : Counts how many times each word appears, Simple & effective : CountVectorizer
# BoW treats all words equally (same importance)

# Create object for CountVectorizer
vectorizer = CountVectorizer(max_features=5000)     # max_features=5000 limits the vocabulary to the top 5000 most frequent words in the dataset. Reduces Vocabulary size and focused learning

# Learn vocabulary and convert text to numbers both at once for review column
X = vectorizer.fit_transform(df['cleaned_review'])

# Target Variable
y = df['sentiment']

#### 2. TF - IDF (Term Frequency - Inverse Document Frequency)

In [27]:
# Solving the limitations of BOW
# Give more importance to rare but meaningful words : TfidfVectorizer

# Create object for TfidfVectorizer
tfidf =  TfidfVectorizer(max_features=5000)

X_tfidf = tfidf.fit_transform(df['cleaned_review'])

print(X_tfidf.shape)

(50000, 5000)


#### Model Building

In [ ]:
# Train Test and Split data
X_train, X_test, y_train, y_test =  train_test_split(X_tfidf, y, test_size=0.2, random_state=42)

print("X_train Shape : ", X_train.shape)
print("X_test Shape : ", X_test.shape)

# These data contains pure numbers E.g. [0.2, 0.0, 0.5, ...] now models can undertstand

X_train Shape :  (40000, 5000)
X_test Shape :  (10000, 5000)


In [38]:
# Logistic Regression : A classification algorithm

# Create model object
model =  LogisticRegression()

# train model on trainind data
# Label encoding is automatic in scikit-learn (for target variable)
model.fit(X_train, y_train)

# Predict 
y_pred = model.predict(X_test)

# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy : ", accuracy)

Accuracy :  0.8876


In [40]:
report= classification_report(y_test, y_pred)
print(report)

              precision    recall  f1-score   support

    negative       0.90      0.87      0.89      4961
    positive       0.88      0.90      0.89      5039

    accuracy                           0.89     10000
   macro avg       0.89      0.89      0.89     10000
weighted avg       0.89      0.89      0.89     10000



In [42]:
# Naive Bayes : Probability Based model, assumes features are independent, fast

# Create model
nb_model = MultinomialNB()

# Train Model
nb_model.fit(X_train, y_train)

# Predict
y_pred_nb = nb_model.predict(X_test)

# Report
report_nb = classification_report(y_pred_nb, y_test)
print(report_nb)

              precision    recall  f1-score   support

    negative       0.85      0.86      0.85      4923
    positive       0.86      0.85      0.86      5077

    accuracy                           0.85     10000
   macro avg       0.85      0.85      0.85     10000
weighted avg       0.85      0.85      0.85     10000



In [44]:
# Decision Tree : splits data based on conditions

# Create Model
dt_model =  DecisionTreeClassifier()

# Train Model
dt_model.fit(X_train, y_train)

# Predict
y_pred_dt = dt_model.predict(X_test)

# Report
report_dt = classification_report(y_pred_dt, y_test)
print(report_dt)

              precision    recall  f1-score   support

    negative       0.72      0.72      0.72      4980
    positive       0.72      0.73      0.72      5020

    accuracy                           0.72     10000
   macro avg       0.72      0.72      0.72     10000
weighted avg       0.72      0.72      0.72     10000



#### Model Evalution

In [46]:
print("Logistic Regression Report :\n",report)
print("\nNaive Bayes Report :\n",report_nb)
print("\nDecision Tree Report :\n",report_dt)

Logistic Regression Report :
               precision    recall  f1-score   support

    negative       0.90      0.87      0.89      4961
    positive       0.88      0.90      0.89      5039

    accuracy                           0.89     10000
   macro avg       0.89      0.89      0.89     10000
weighted avg       0.89      0.89      0.89     10000


Naive Bayes Report :
               precision    recall  f1-score   support

    negative       0.85      0.86      0.85      4923
    positive       0.86      0.85      0.86      5077

    accuracy                           0.85     10000
   macro avg       0.85      0.85      0.85     10000
weighted avg       0.85      0.85      0.85     10000


Decision Tree Report :
               precision    recall  f1-score   support

    negative       0.72      0.72      0.72      4980
    positive       0.72      0.73      0.72      5020

    accuracy                           0.72     10000
   macro avg       0.72      0.72      0.72     10

#### Comparison & Insights

1. Best Preprocessing

- Lowercasing, stopword removal, URL removal, and lemmatization helped clean the raw and noisy text, improving overall text quality. Additionally, preserving important words like "no" and "not" ensured that the sentiment of the sentence was not lost.

2. Best Feature Engineering

- TF-IDF performed better than Bag of Words because it assigns higher importance to meaningful and rare words while reducing the importance of common words. This helps overcome the limitation of Bag of Words, which treats all words equally regardless of their significance.

3. Best Model

- Logistic Regression performed the best among all models due to its ability to handle high-dimensional sparse data efficiently and capture relationships between features. It provided better accuracy and generalization compared to Naive Bayes and Decision Tree.

4. Trade-offs
- Naive Bayes: Fast and efficient, but assumes independence between words, which is not realistic in natural language.
- Logistic Regression: More accurate and robust, but slightly slower compared to Naive Bayes.
- Decision Tree: Easy to understand and interpret, but prone to overfitting and not well-suited for high-dimensional text data.

#### Testing with an Example

In [47]:
# Create a function for our model to predict any input text
def sentiment_analizer(text):
    clean_text = preprocess_text(text)
    vector = tfidf.transform([clean_text])
    prediction = model.predict(vector)
    print("Review : ", text)
    print("Sentiment : ",prediction)


In [52]:
sentiment_analizer("It was a very bad film !!")

Review :  It was a very bad film !!
Sentiment :  ['negative']


In [56]:
sentiment_analizer("Why are you happy ??")

Review :  Why are you happy ??
Sentiment :  ['positive']
